# Cell 01 - Load corpus chunks để chuẩn bị build Dense Vector Index

## Mục tiêu cell này
Đọc lại file `all_chunks.csv` để chuẩn bị tạo embedding cho toàn bộ corpus RAG.

## Vì sao cần làm bước này?
BM25 tìm kiếm dựa trên từ khóa, còn Dense Retrieval tìm kiếm dựa trên ngữ nghĩa.

Ví dụ:
- BM25 mạnh khi câu hỏi và tài liệu có nhiều từ khóa giống nhau.
- Dense Retrieval mạnh hơn khi người dùng hỏi khác cách diễn đạt nhưng vẫn cùng ý nghĩa.

Trong project này, Dense Retrieval sẽ giúp cải thiện khả năng tìm đúng evidence, đặc biệt khi câu hỏi không trùng từ khóa chính xác với tài liệu.

## Giải thích code
Code sẽ:
1. Xác định đúng thư mục project
2. Đọc file `data/chunks/all_chunks.csv`
3. Chuẩn hóa kiểu dữ liệu quan trọng
4. Tạo thư mục `indexes/faiss`
5. Kiểm tra số lượng chunk và phân bố nguồn dữ liệu

## Output mong đợi
Bạn cần thấy:
- tổng corpus khoảng 91,448 chunks
- source gồm `legal` và `company_handbook`
- cột `retrieval_text` đã sẵn sàng để tạo embedding

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import pickle
import os

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

CHUNK_DIR = PROJECT / "data" / "chunks"
FAISS_DIR = PROJECT / "indexes" / "faiss"

FAISS_DIR.mkdir(parents=True, exist_ok=True)

all_chunks_df = pd.read_csv(CHUNK_DIR / "all_chunks.csv", low_memory=False)

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    all_chunks_df[col] = all_chunks_df[col].fillna("").astype(str)

all_chunks_df["retrieval_text"] = all_chunks_df["retrieval_text"].fillna("").astype(str)
all_chunks_df["chunk_text"] = all_chunks_df["chunk_text"].fillna("").astype(str)
all_chunks_df["chunk_chars"] = pd.to_numeric(all_chunks_df["chunk_chars"], errors="coerce").fillna(0).astype(int)

print("Project:", PROJECT)
print("All chunks shape:", all_chunks_df.shape)

print("\nSource distribution:")
display(all_chunks_df["source_type"].value_counts().reset_index())

display(
    all_chunks_df[
        ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "chunk_chars", "language"]
    ].head()
)

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
All chunks shape: (91448, 11)

Source distribution:


,source_type,count
0,legal,91353
1,company_handbook,95


,corpus_id,chunk_id,parent_id,source_type,title,chunk_chars,language
0,chunk_000000,legal_142820_000,142820,legal,legal_cid_142820,767,Vietnamese
1,chunk_000001,legal_27817_000,27817,legal,legal_cid_27817,1194,Vietnamese
2,chunk_000002,legal_27817_001,27817,legal,legal_cid_27817,212,Vietnamese
3,chunk_000003,legal_72117_000,72117,legal,legal_cid_72117,811,Vietnamese
4,chunk_000004,legal_33215_000,33215,legal,legal_cid_33215,394,Vietnamese


# Cell 02 - Kiểm tra thư viện embedding, FAISS và GPU

## Mục tiêu cell này
Kiểm tra môi trường hiện tại đã sẵn sàng để build Dense Vector Index hay chưa.

## Vì sao cần làm bước này?
Khác với BM25, Dense Retrieval cần biến mỗi chunk văn bản thành vector embedding.  
Việc tạo embedding cho 91,448 chunks có thể khá nặng, nên nếu máy có GPU thì nên dùng GPU để tăng tốc.

Trong notebook này, ta cần kiểm tra:
- `torch`: để xem có CUDA/GPU không
- `sentence_transformers`: để load embedding model
- `faiss`: để build vector index

## Giải thích code
Code sẽ:
1. Import các thư viện cần dùng
2. Kiểm tra GPU CUDA có khả dụng không
3. In ra thiết bị sẽ dùng: `cuda` hoặc `cpu`
4. In version của torch và faiss nếu có

## Output mong đợi
Nếu máy nhận GPU, bạn sẽ thấy:
`Device: cuda`

Nếu không nhận GPU, bạn sẽ thấy:
`Device: cpu`

CPU vẫn chạy được, nhưng tạo embedding sẽ chậm hơn.

In [2]:
import torch
import faiss
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("FAISS version:", faiss.__version__)
print("SentenceTransformer import: OK")

W0509 03:51:39.240000 21200 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Torch version: 2.7.1+cu118
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
FAISS version: 1.12.0
SentenceTransformer import: OK


# Cell 03 - Load multilingual embedding model cho Dense Retrieval

## Mục tiêu cell này
Load model embedding đa ngôn ngữ để tạo vector cho toàn bộ corpus RAG.

## Vì sao cần làm bước này?
BM25 ở notebook 04 chỉ tìm theo từ khóa.  
Dense Retrieval sẽ tìm theo ngữ nghĩa, tức là có thể tìm tài liệu liên quan dù câu hỏi và tài liệu không trùng từ khóa hoàn toàn.

Project này có hai nguồn dữ liệu:
- pháp luật tiếng Việt
- handbook nội bộ công ty tiếng Anh

Vì vậy ta dùng model multilingual để hỗ trợ cả tiếng Việt và tiếng Anh.

## Model được dùng
`intfloat/multilingual-e5-base`

Với dòng E5:
- tài liệu/chunk nên thêm prefix `passage:`
- câu hỏi nên thêm prefix `query:`

Việc này giúp embedding phù hợp hơn cho bài toán retrieval.

## Giải thích code
Code sẽ:
1. Khai báo tên model embedding
2. Load model bằng `SentenceTransformer`
3. Đưa model lên GPU nếu có CUDA
4. In ra thông tin model và device đang dùng

## Output mong đợi
Bạn cần thấy model load thành công và device là `cuda`.

In [3]:
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Device:", embedding_model.device)
print("Max sequence length:", embedding_model.max_seq_length)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

C:\Users\npd20\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\npd20\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model: intfloat/multilingual-e5-base
Device: cuda:0
Max sequence length: 512


# Cell 04 - Test tạo embedding mẫu bằng multilingual-e5-base

## Mục tiêu cell này
Kiểm tra model embedding đã tạo vector đúng chưa trước khi encode toàn bộ 91,448 chunks.

## Vì sao cần làm bước này?
Tạo embedding cho toàn bộ corpus là bước nặng và mất thời gian.  
Trước khi chạy full, ta nên test trên vài câu mẫu để đảm bảo:
- model chạy được trên GPU
- output là ma trận vector đúng shape
- vector đã được normalize để dùng với FAISS cosine similarity

## Lưu ý với E5 model
Dòng E5 yêu cầu thêm prefix:
- `query:` cho câu hỏi
- `passage:` cho tài liệu/chunk

Ví dụ:
- `query: Lương thử việc được quy định như thế nào?`
- `passage: Điều 2. ...`

## Giải thích code
Code sẽ:
1. Tạo 2 passage mẫu từ corpus
2. Tạo 1 query mẫu
3. Encode bằng `embedding_model.encode()`
4. Bật `normalize_embeddings=True` để vector phù hợp cho cosine similarity
5. In ra shape embedding

## Output mong đợi
Bạn cần thấy embedding shape dạng `(3, 768)` với model `multilingual-e5-base`.

In [4]:
sample_texts = [
    "query: Lương thử việc được quy định như thế nào?",
    "passage: " + all_chunks_df.loc[0, "retrieval_text"],
    "passage: " + all_chunks_df[all_chunks_df["source_type"] == "company_handbook"].iloc[0]["retrieval_text"]
]

sample_embeddings = embedding_model.encode(
    sample_texts,
    batch_size=3,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Sample embedding shape:", sample_embeddings.shape)
print("Embedding dtype:", sample_embeddings.dtype)
print("Vector norm mẫu:", round(float(np.linalg.norm(sample_embeddings[0])), 4))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Sample embedding shape: (3, 768)
Embedding dtype: float32
Vector norm mẫu: 1.0


# Cell 05 - Chuẩn bị văn bản corpus để tạo dense embeddings

## Mục tiêu cell này
Chuẩn bị danh sách văn bản đầu vào cho embedding model `intfloat/multilingual-e5-base`.

## Vì sao cần làm bước này?
Dense Retrieval không tìm kiếm bằng từ khóa như BM25.  
Nó biến mỗi chunk thành vector embedding để tìm các đoạn có ý nghĩa gần với câu hỏi.

Với model E5, tài liệu cần thêm prefix `passage:` trước nội dung.  
Nếu không thêm prefix này, chất lượng retrieval có thể giảm vì model được huấn luyện theo format query/passage.

## Giải thích code
Code sẽ:
1. Lấy cột `retrieval_text` từ `all_chunks_df`
2. Thêm prefix `passage:` cho mỗi chunk
3. Kiểm tra số lượng văn bản cần encode
4. Hiển thị thử 1 legal passage và 1 company handbook passage

## Output mong đợi
Bạn cần thấy:
- số passage bằng 91,448
- ví dụ passage bắt đầu bằng `passage:`

In [5]:
passage_texts = [
    "passage: " + text
    for text in all_chunks_df["retrieval_text"].tolist()
]

print("Số passages cần encode:", len(passage_texts))
print("Số chunks trong dataframe:", len(all_chunks_df))

print("\nVí dụ legal passage:")
print(passage_texts[0][:700])

company_idx = all_chunks_df[all_chunks_df["source_type"] == "company_handbook"].index[0]

print("\nVí dụ company handbook passage:")
print(passage_texts[company_idx][:700])

Số passages cần encode: 91448
Số chunks trong dataframe: 91448

Ví dụ legal passage:
passage: legal_cid_142820
“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư Việt Nam
1. Liên đoàn Luật sư Việt Nam là tổ chức xã hội - nghề nghiệp thống nhất trong toàn quốc của các Đoàn Luật sư, các luật sư Việt Nam; có tư cách pháp nhân, có con dấu, tài khoản.
2. Biểu tượng của Liên đoàn Luật sư Việt Nam là hình tròn nền xanh da trời, chính giữa là cán cân công lý gắn với hình tượng cuốn sách, dưới cán cân công lý là dòng chữ “VIETNAM BAR FEDERATION", hai bên mỗi bên có ba dải màu vàng đậm, phía trên là ngôi sao vàng hình cờ Tổ quốc Việt Nam và dòng chữ Liên đoàn Luật sư Việt Nam.
3. Tên giao dịch quốc tế của Liên đoàn Luật sư Việt Nam là Vietnam Bar Federation (viết tắt là VBF).
4. Trụ sở của

Ví dụ company handbook passage:
passage: Benefits And Perks
# Benefits & Perks
## Health Insurance
Detailed information about all 37signals insurance policies and other benefits can be found in Basecamp.

### Medi

# Cell 06 - Tạo dense embeddings cho toàn bộ corpus

## Mục tiêu cell này
Tạo vector embedding cho toàn bộ 91,448 chunks trong `all_chunks.csv`.

## Vì sao cần làm bước này?
BM25 tìm kiếm bằng từ khóa, còn Dense Retrieval tìm kiếm bằng ngữ nghĩa.  
Để tìm kiếm ngữ nghĩa, mỗi chunk cần được chuyển thành một vector số.

Ví dụ:
- Câu hỏi: `laptop công ty khi nghỉ việc xử lý sao?`
- Tài liệu có thể viết: `managing work devices after separation`

Dù không trùng từ khóa hoàn toàn, dense embedding vẫn có thể giúp tìm được nội dung gần nghĩa.

## Giải thích code
Code sẽ:
1. Dùng model `intfloat/multilingual-e5-base` để encode toàn bộ `passage_texts`
2. Bật `normalize_embeddings=True` để dùng cosine similarity với FAISS
3. Lưu embeddings vào `indexes/faiss/corpus_embeddings.npy`
4. Lưu metadata tương ứng vào `indexes/faiss/dense_metadata.csv`

## Output mong đợi
Bạn cần thấy embedding shape khoảng `(91448, 768)`.  
File `.npy` sẽ có dung lượng khoảng vài trăm MB.

In [6]:
embeddings_path = FAISS_DIR / "corpus_embeddings.npy"
metadata_path = FAISS_DIR / "dense_metadata.csv"

if embeddings_path.exists():
    corpus_embeddings = np.load(embeddings_path)
    print("Đã tồn tại embeddings, load lại từ file.")
else:
    corpus_embeddings = embedding_model.encode(
        passage_texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save(embeddings_path, corpus_embeddings)

all_chunks_df.to_csv(metadata_path, index=False, encoding="utf-8-sig")

print("Embeddings shape:", corpus_embeddings.shape)
print("Embeddings dtype:", corpus_embeddings.dtype)
print("Đã lưu embeddings:", embeddings_path)
print("Đã lưu metadata:", metadata_path)
print("Embeddings size MB:", round(embeddings_path.stat().st_size / 1024 / 1024, 2))

Batches:   0%|          | 0/2858 [00:00<?, ?it/s]

Embeddings shape: (91448, 768)
Embeddings dtype: float32
Đã lưu embeddings: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\indexes\faiss\corpus_embeddings.npy
Đã lưu metadata: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\indexes\faiss\dense_metadata.csv
Embeddings size MB: 267.91


# Cell 07 - Build FAISS Dense Vector Index

## Mục tiêu cell này
Xây dựng FAISS index từ toàn bộ dense embeddings đã tạo ở Cell 06.

## Vì sao cần làm bước này?
Ở Cell 06, mỗi chunk đã được chuyển thành vector embedding.  
Nhưng nếu chỉ lưu embeddings dạng `.npy`, việc tìm kiếm sẽ chậm vì phải so sánh thủ công query với toàn bộ 91,448 vectors.

FAISS giúp tạo một vector index để tìm kiếm nhanh hơn.

Trong project này:
- BM25 index dùng để tìm theo từ khóa
- FAISS index dùng để tìm theo ngữ nghĩa
- Sau này Hybrid RAG sẽ kết hợp cả hai

## Giải thích code
Code sẽ:
1. Lấy số chiều embedding, ở đây là 768
2. Tạo FAISS index dạng `IndexFlatIP`
3. Vì embeddings đã được normalize, inner product tương đương cosine similarity
4. Add toàn bộ corpus embeddings vào FAISS index
5. Lưu index vào `indexes/faiss/faiss_index.index`

## Output mong đợi
Bạn cần thấy:
- embedding dimension = 768
- số vector trong FAISS index = 91,448
- file FAISS index được lưu thành công

In [7]:
faiss_index_path = FAISS_DIR / "faiss_index.index"

embedding_dim = corpus_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(corpus_embeddings)

faiss.write_index(faiss_index, str(faiss_index_path))

print("Embedding dimension:", embedding_dim)
print("FAISS index vectors:", faiss_index.ntotal)
print("Đã lưu FAISS index:", faiss_index_path)
print("FAISS index size MB:", round(faiss_index_path.stat().st_size / 1024 / 1024, 2))

Embedding dimension: 768
FAISS index vectors: 91448
Đã lưu FAISS index: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\indexes\faiss\faiss_index.index
FAISS index size MB: 267.91


# Cell 08 - Tạo hàm Dense Search và kiểm tra truy xuất thử

## Mục tiêu cell này
Tạo hàm tìm kiếm bằng FAISS Dense Index và chạy thử trên một vài câu hỏi mẫu.

## Vì sao cần làm bước này?
Sau khi build FAISS index, ta cần kiểm tra hệ thống có thể:
- nhận một câu hỏi người dùng
- biến câu hỏi thành embedding vector
- tìm các chunk có embedding gần nhất trong FAISS index
- trả về nguồn tài liệu liên quan

Khác với BM25, Dense Retrieval không chỉ dựa vào trùng từ khóa mà dựa vào độ gần ngữ nghĩa giữa câu hỏi và tài liệu.

## Giải thích code
Code sẽ:
1. Định nghĩa hàm `dense_search()`
2. Thêm prefix `query:` cho câu hỏi theo chuẩn E5
3. Encode câu hỏi thành vector embedding
4. Search trong FAISS index
5. Trả về top-k chunk liên quan nhất kèm similarity score
6. Test một câu hỏi pháp luật tiếng Việt và một câu hỏi handbook tiếng Anh

## Output mong đợi
Bạn cần thấy bảng kết quả gồm:
- rank
- score
- source_type
- title
- chunk_text

Legal query nên trả về các chunk pháp luật.
Company query nên trả về các chunk handbook nội bộ.

In [8]:
def dense_search(query, top_k=5, source_filter=None):
    query_text = "query: " + str(query)

    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, min(top_k * 10, len(all_chunks_df)))

    rows = []
    rank = 1

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        row = all_chunks_df.iloc[idx]

        if source_filter is not None and row["source_type"] not in source_filter:
            continue

        item = row.to_dict()
        item["score"] = float(score)
        item["rank"] = rank
        rows.append(item)
        rank += 1

        if len(rows) >= top_k:
            break

    results = pd.DataFrame(rows)

    return results[
        [
            "rank",
            "score",
            "corpus_id",
            "chunk_id",
            "parent_id",
            "source_type",
            "title",
            "language",
            "chunk_text"
        ]
    ]

legal_query = "Lương thử việc được quy định như thế nào?"
company_query = "What is the company policy for managing work devices?"

print("Dense test - Legal query:")
display(dense_search(legal_query, top_k=5, source_filter=["legal"]))

print("\nDense test - Company handbook query:")
display(dense_search(company_query, top_k=5, source_filter=["company_handbook"]))

Dense test - Legal query:


,rank,score,corpus_id,chunk_id,parent_id,source_type,title,language,chunk_text
0,1,0.876999,chunk_073704,legal_116782_000,116782,legal,legal_cid_116782,Vietnamese,Vi phạm quy định về thử việc\n1. Phạt tiền từ ...
1,2,0.874766,chunk_002341,legal_61953_000,61953,legal,legal_cid_61953,Vietnamese,“Điều 26. Tiền lương trong thời gian thử việc\...
2,3,0.873931,chunk_047528,legal_172654_001,172654,legal,legal_cid_172654,Vietnamese,"bắt buộc, bảo hiểm y tế, bảo hiểm thất nghiệp...."
3,4,0.871711,chunk_035605,legal_175106_000,175106,legal,legal_cid_175106,Vietnamese,Thử việc và thời gian thử việc\n1. Thử việc\na...
4,5,0.867452,chunk_001611,legal_36069_000,36069,legal,legal_cid_36069,Vietnamese,"Mức phạt tiền, thẩm quyền xử phạt và nguyên tắ..."



Dense test - Company handbook query:


,rank,score,corpus_id,chunk_id,parent_id,source_type,title,language,chunk_text
0,1,0.825828,chunk_091383,company_0004_001,company_0004,company_handbook,Managing Work Devices,English,"With Linux, we run Omarchy, which already come..."
1,2,0.825389,chunk_091382,company_0004_000,company_0004,company_handbook,Managing Work Devices,English,# Managing work devices\nEveryone receives a n...
2,3,0.819602,chunk_091384,company_0004_002,company_0004,company_handbook,Managing Work Devices,English,## Mobile devices and Windows\nDevices running...
3,4,0.788421,chunk_091387,company_0005_002,company_0005,company_handbook,Moonlighting,English,## Not OK\n1. You can’t work full time or part...
4,5,0.783055,chunk_091394,company_0008_000,company_0008,company_handbook,Readme,English,Read on basecamp.com\n\n# 37signals Employee H...


# Cell 09 - Load validation/test split để đánh giá Dense Retrieval

## Mục tiêu cell này
Đọc lại `val_strict.csv` và `test_final.csv` để chuẩn bị đánh giá Dense Retrieval.

## Vì sao cần làm bước này?
Ở các cell trước, ta đã test Dense Search bằng vài câu hỏi mẫu.  
Tuy nhiên, để biết Dense Retrieval thật sự tốt hay không, ta cần đánh giá trên validation set giống như đã làm với BM25.

Validation set dùng để đo:
- Dense Retrieval tìm đúng tài liệu trong top-k tốt không
- Dense có cải thiện so với BM25 không
- Sau này có nên kết hợp Dense với BM25 trong Hybrid Retrieval không

Test set vẫn chỉ dùng cho đánh giá cuối cùng, chưa dùng ở bước này.

## Giải thích code
Code sẽ:
1. Đọc `val_strict.csv`
2. Đọc `test_final.csv`
3. Chuyển `relevant_cids` từ chuỗi JSON về list Python
4. In ra shape để xác nhận dữ liệu đã load đúng

## Output mong đợi
Bạn cần thấy:
- validation khoảng 8,605 dòng
- test khoảng 29,746 dòng
- `relevant_cids` hiển thị dạng list như `[243090]`

In [9]:
SPLIT_DIR = PROJECT / "data" / "splits"

val_df = pd.read_csv(SPLIT_DIR / "val_strict.csv")
test_df = pd.read_csv(SPLIT_DIR / "test_final.csv")

val_df["relevant_cids"] = val_df["relevant_cids"].apply(json.loads)
test_df["relevant_cids"] = test_df["relevant_cids"].apply(json.loads)

print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(val_df.head())

Validation shape: (8605, 4)
Test shape: (29746, 4)


,qid,question,relevant_cids,num_relevant
0,162228,Chế độ báo cáo của Giám đốc Ban Quản lý dự án ...,[243090],1
1,3414,Việc tích nước vận hành và tích nước thời kỳ t...,[65231],1
2,108965,Phanh cần trục khi dừng khẩn cấp phải là loại ...,[183501],1
3,103161,Ai có trách nhiệm báo cáo kết quả đánh giá ngh...,[177010],1
4,70371,Làm sao để được công nhận văn bằng nước ngoài ...,"[140295, 140296]",2


# Cell 10 - Tạo hàm Dense Retrieval và metric đánh giá

## Mục tiêu cell này
Tạo hàm truy xuất Dense Retrieval trả về `parent_id/cid`, đồng thời tạo lại các metric đánh giá retrieval.

## Vì sao cần làm bước này?
FAISS trả về các `chunk_id`, nhưng ground truth của dataset nằm ở dạng `relevant_cids`.

Ví dụ:
- FAISS trả về chunk: `legal_61953_000`
- Chunk này có `parent_id = 61953`
- Nếu `61953` nằm trong `relevant_cids` thì xem là truy xuất đúng.

Các metric dùng để đánh giá gồm:
- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR
- nDCG@10

## Giải thích code
Code sẽ:
1. Định nghĩa lại các hàm metric retrieval
2. Tạo hàm `dense_retrieve_parent_ids()`
3. Encode câu hỏi với prefix `query:`
4. Search bằng FAISS
5. Trả về danh sách `parent_id` không trùng lặp
6. Test thử trên 3 câu validation đầu tiên

## Output mong đợi
Bạn cần thấy mỗi sample có:
- câu hỏi
- relevant cids
- dense predicted parent ids
- metric tương ứng

In [10]:
import math

def recall_at_k(pred_ids, relevant_ids, k):
    return 1.0 if set(pred_ids[:k]) & set(relevant_ids) else 0.0

def mrr_score(pred_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    for rank, pred_id in enumerate(pred_ids, start=1):
        if pred_id in relevant_set:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(pred_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    dcg = 0.0

    for i, pred_id in enumerate(pred_ids[:k], start=1):
        rel = 1.0 if pred_id in relevant_set else 0.0
        dcg += rel / math.log2(i + 1)

    ideal_hits = min(len(relevant_set), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0

def compute_retrieval_metrics(pred_ids, relevant_ids):
    return {
        "recall@1": recall_at_k(pred_ids, relevant_ids, 1),
        "recall@3": recall_at_k(pred_ids, relevant_ids, 3),
        "recall@5": recall_at_k(pred_ids, relevant_ids, 5),
        "recall@10": recall_at_k(pred_ids, relevant_ids, 10),
        "mrr": mrr_score(pred_ids, relevant_ids),
        "ndcg@10": ndcg_at_k(pred_ids, relevant_ids, 10)
    }

def dense_retrieve_parent_ids(query, top_k=10, source_filter=["legal"], candidate_multiplier=5):
    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    search_k = min(top_k * candidate_multiplier * 10, len(all_chunks_df))
    scores, indices = faiss_index.search(query_embedding, search_k)

    parent_ids = []
    seen = set()

    for idx in indices[0]:
        if idx == -1:
            continue

        row = all_chunks_df.iloc[idx]

        if source_filter is not None and row["source_type"] not in source_filter:
            continue

        parent_id = str(row["parent_id"])

        if parent_id not in seen:
            parent_ids.append(parent_id)
            seen.add(parent_id)

        if len(parent_ids) >= top_k:
            break

    return parent_ids

for i in range(3):
    row = val_df.iloc[i]
    pred_ids = dense_retrieve_parent_ids(row["question"], top_k=10)
    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    print("\n--- Sample", i, "---")
    print("Question:", row["question"])
    print("Relevant cids:", relevant_ids)
    print("Dense pred parent_ids:", pred_ids[:10])
    print("Metrics:", metrics)


--- Sample 0 ---
Question: Chế độ báo cáo của Giám đốc Ban Quản lý dự án đầu tư xây dựng chuyên ngành thuộc Kiểm toán nhà nước được quy định như thế nào?
Relevant cids: ['243090']
Dense pred parent_ids: ['98280', '243090', '61305', '158058', '68109', '115404', '104253', '115970', '115405', '23766']
Metrics: {'recall@1': 0.0, 'recall@3': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 0.5, 'ndcg@10': 0.6309297535714575}

--- Sample 1 ---
Question: Việc tích nước vận hành và tích nước thời kỳ thi công đối với công trình thủy lợi đập đất đầm nén được quy định như thế nào?
Relevant cids: ['65231']
Dense pred parent_ids: ['65231', '65230', '231128', '105613', '42184', '13982', '140701', '140700', '74681', '182852']
Metrics: {'recall@1': 1.0, 'recall@3': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 1.0, 'ndcg@10': 1.0}

--- Sample 2 ---
Question: Phanh cần trục khi dừng khẩn cấp phải là loại phanh như thế nào?
Relevant cids: ['183501']
Dense pred parent_ids: ['183501', '183502', '38369', '

# Cell 11 - Đánh giá Dense Retrieval trên validation set

## Mục tiêu cell này
Đánh giá Dense Retrieval trên toàn bộ `val_strict.csv` để so sánh với BM25 baseline.

## Vì sao cần làm bước này?
Ở notebook 04, BM25 đạt Recall@10 khoảng 68.7%.  
Bây giờ ta cần đánh giá Dense Retrieval để xem tìm kiếm ngữ nghĩa có tốt hơn tìm kiếm từ khóa hay không.

Dense Retrieval có thể mạnh hơn BM25 trong các trường hợp:
- câu hỏi diễn đạt khác tài liệu
- tài liệu liên quan về mặt ý nghĩa nhưng không trùng nhiều từ khóa
- nhiều văn bản có từ khóa giống nhau nhưng nội dung khác nhau

## Giải thích code
Code sẽ:
1. Thêm prefix `query:` cho toàn bộ câu hỏi validation
2. Encode toàn bộ câu hỏi thành query embeddings bằng GPU
3. Search trong FAISS index để lấy top candidates
4. Lọc `source_type = legal` vì validation benchmark dùng legal cid
5. Quy kết quả chunk về `parent_id/cid`
6. Tính Recall@1, Recall@3, Recall@5, Recall@10, MRR, nDCG@10
7. Lưu predictions và metrics vào thư mục `artifacts`

## Output mong đợi
Bạn cần thấy bảng metric Dense Retrieval trên validation set để so sánh với BM25.

In [11]:
from tqdm.auto import tqdm

METRIC_DIR = PROJECT / "artifacts" / "metrics"
PRED_DIR = PROJECT / "artifacts" / "predictions"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

DENSE_TOP_K = 10
SEARCH_K = 500

query_texts = ["query: " + str(q) for q in val_df["question"].tolist()]

query_embeddings = embedding_model.encode(
    query_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

scores, indices = faiss_index.search(query_embeddings, SEARCH_K)

dense_val_rows = []

for row_idx, row in tqdm(list(val_df.iterrows()), total=len(val_df)):
    pred_ids = []
    seen = set()

    for idx in indices[row_idx]:
        if idx == -1:
            continue

        chunk_row = all_chunks_df.iloc[idx]

        if chunk_row["source_type"] != "legal":
            continue

        parent_id = str(chunk_row["parent_id"])

        if parent_id not in seen:
            pred_ids.append(parent_id)
            seen.add(parent_id)

        if len(pred_ids) >= DENSE_TOP_K:
            break

    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    dense_val_rows.append({
        "qid": row["qid"],
        "question": row["question"],
        "relevant_cids": json.dumps(relevant_ids, ensure_ascii=False),
        "pred_parent_ids": json.dumps(pred_ids, ensure_ascii=False),
        **metrics
    })

dense_val_results_df = pd.DataFrame(dense_val_rows)

dense_val_summary_df = (
    dense_val_results_df[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
    ]
    .mean()
    .reset_index()
)

dense_val_summary_df.columns = ["metric", "value"]
dense_val_summary_df["method"] = "Dense_E5"

results_path = PRED_DIR / "dense_val_predictions.csv"
summary_path = METRIC_DIR / "dense_val_metrics.csv"

dense_val_results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
dense_val_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Đã lưu predictions:", results_path)
print("Đã lưu metrics:", summary_path)

display(dense_val_summary_df)

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

  0%|          | 0/8605 [00:00<?, ?it/s]

Đã lưu predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\dense_val_predictions.csv
Đã lưu metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\dense_val_metrics.csv


,metric,value,method
0,recall@1,0.421615,Dense_E5
1,recall@3,0.624869,Dense_E5
2,recall@5,0.700407,Dense_E5
3,recall@10,0.783614,Dense_E5
4,mrr,0.540115,Dense_E5
5,ndcg@10,0.585338,Dense_E5


# Cell 12 - So sánh BM25 và Dense Retrieval trên validation set

## Mục tiêu cell này
So sánh kết quả retrieval của BM25 và Dense Retrieval trên cùng validation set.

## Vì sao cần làm bước này?
BM25 là baseline tìm kiếm theo từ khóa.  
Dense Retrieval là phương pháp tìm kiếm theo ngữ nghĩa.

Việc so sánh hai phương pháp giúp trả lời câu hỏi:
- Dense Retrieval có cải thiện so với BM25 không?
- Metric nào được cải thiện nhiều nhất?
- Có nên tiếp tục xây Hybrid Retrieval không?

Nếu Dense tốt hơn BM25 nhưng BM25 vẫn có điểm mạnh riêng, bước tiếp theo sẽ là kết hợp cả hai bằng Hybrid Retrieval.

## Giải thích code
Code sẽ:
1. Đọc file `bm25_val_metrics.csv`
2. Đọc file `dense_val_metrics.csv`
3. Gộp hai bảng metric
4. Pivot bảng để dễ so sánh BM25 và Dense
5. Tính mức cải thiện tuyệt đối của Dense so với BM25
6. Lưu bảng so sánh vào `artifacts/metrics/bm25_vs_dense_val_metrics.csv`

## Output mong đợi
Bạn cần thấy bảng so sánh gồm các metric:
- recall@1
- recall@3
- recall@5
- recall@10
- mrr
- ndcg@10

Cột `improvement_dense_minus_bm25` cho biết Dense tốt hơn BM25 bao nhiêu.

In [12]:
bm25_metrics_df = pd.read_csv(METRIC_DIR / "bm25_val_metrics.csv")
dense_metrics_df = pd.read_csv(METRIC_DIR / "dense_val_metrics.csv")

compare_metrics_df = pd.concat([bm25_metrics_df, dense_metrics_df], ignore_index=True)

pivot_df = compare_metrics_df.pivot(
    index="metric",
    columns="method",
    values="value"
).reset_index()

pivot_df["improvement_dense_minus_bm25"] = pivot_df["Dense_E5"] - pivot_df["BM25"]
pivot_df["relative_improvement_percent"] = (
    pivot_df["improvement_dense_minus_bm25"] / pivot_df["BM25"] * 100
).round(2)

pivot_df["BM25"] = pivot_df["BM25"].round(6)
pivot_df["Dense_E5"] = pivot_df["Dense_E5"].round(6)
pivot_df["improvement_dense_minus_bm25"] = pivot_df["improvement_dense_minus_bm25"].round(6)

compare_path = METRIC_DIR / "bm25_vs_dense_val_metrics.csv"
pivot_df.to_csv(compare_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", compare_path)
display(pivot_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\bm25_vs_dense_val_metrics.csv


method,metric,BM25,Dense_E5,improvement_dense_minus_bm25,relative_improvement_percent
0,mrr,0.453844,0.540115,0.086271,19.01
1,ndcg@10,0.497519,0.585338,0.087819,17.65
2,recall@1,0.343754,0.421615,0.077862,22.65
3,recall@10,0.687391,0.783614,0.096223,14.00
4,recall@3,0.528995,0.624869,0.095874,18.12
5,recall@5,0.605113,0.700407,0.095293,15.75


# Cell 13 - Phân tích ví dụ Dense Retrieval cải thiện so với BM25

## Mục tiêu cell này
Tìm các câu hỏi mà BM25 không tìm được tài liệu đúng trong top 10, nhưng Dense Retrieval tìm được trong top 10.

## Vì sao cần làm bước này?
Bảng metric cho biết Dense tốt hơn BM25 về số liệu, nhưng khi thuyết trình hoặc viết báo cáo, ta cần ví dụ cụ thể để giải thích.

Ví dụ:
- BM25 có thể sai vì câu hỏi và tài liệu không trùng từ khóa nhiều.
- Dense Retrieval có thể đúng vì nó hiểu gần nghĩa giữa câu hỏi và tài liệu.

Những ví dụ này sẽ chứng minh vì sao project cần Dense Retrieval và sau này cần Hybrid Retrieval.

## Giải thích code
Code sẽ:
1. Đọc predictions của BM25 và Dense trên validation set
2. Chuyển `relevant_cids` và `pred_parent_ids` từ chuỗi JSON về list
3. Tạo cột `hit@10` cho từng phương pháp
4. Tìm các mẫu:
   - BM25 sai nhưng Dense đúng
   - BM25 đúng nhưng Dense sai
   - cả hai cùng đúng
5. Hiển thị một vài ví dụ để phân tích

## Output mong đợi
Bạn cần thấy số lượng các trường hợp:
- BM25 sai nhưng Dense đúng
- BM25 đúng nhưng Dense sai
- cả hai cùng đúng

In [13]:
bm25_pred_df = pd.read_csv(PRED_DIR / "bm25_val_predictions.csv")
dense_pred_df = pd.read_csv(PRED_DIR / "dense_val_predictions.csv")

def parse_list_col(df, col):
    return df[col].apply(json.loads)

def has_hit(pred_ids, relevant_ids, k=10):
    return int(bool(set(pred_ids[:k]) & set(relevant_ids)))

bm25_pred_df["relevant_list"] = parse_list_col(bm25_pred_df, "relevant_cids")
bm25_pred_df["bm25_pred_list"] = parse_list_col(bm25_pred_df, "pred_parent_ids")
bm25_pred_df["bm25_hit@10"] = bm25_pred_df.apply(
    lambda r: has_hit(r["bm25_pred_list"], r["relevant_list"], 10),
    axis=1
)

dense_pred_df["dense_pred_list"] = parse_list_col(dense_pred_df, "pred_parent_ids")
dense_pred_df["dense_hit@10"] = dense_pred_df.apply(
    lambda r: has_hit(r["dense_pred_list"], json.loads(r["relevant_cids"]), 10),
    axis=1
)

compare_pred_df = bm25_pred_df[
    ["qid", "question", "relevant_cids", "pred_parent_ids", "bm25_hit@10"]
].rename(columns={"pred_parent_ids": "bm25_pred_parent_ids"}).merge(
    dense_pred_df[
        ["qid", "pred_parent_ids", "dense_hit@10"]
    ].rename(columns={"pred_parent_ids": "dense_pred_parent_ids"}),
    on="qid",
    how="inner"
)

bm25_wrong_dense_right = compare_pred_df[
    (compare_pred_df["bm25_hit@10"] == 0) & (compare_pred_df["dense_hit@10"] == 1)
].copy()

bm25_right_dense_wrong = compare_pred_df[
    (compare_pred_df["bm25_hit@10"] == 1) & (compare_pred_df["dense_hit@10"] == 0)
].copy()

both_right = compare_pred_df[
    (compare_pred_df["bm25_hit@10"] == 1) & (compare_pred_df["dense_hit@10"] == 1)
].copy()

print("BM25 sai nhưng Dense đúng:", len(bm25_wrong_dense_right))
print("BM25 đúng nhưng Dense sai:", len(bm25_right_dense_wrong))
print("Cả BM25 và Dense cùng đúng:", len(both_right))

print("\nVí dụ BM25 sai nhưng Dense đúng:")
display(
    bm25_wrong_dense_right[
        ["qid", "question", "relevant_cids", "bm25_pred_parent_ids", "dense_pred_parent_ids"]
    ].head(5)
)

print("\nVí dụ BM25 đúng nhưng Dense sai:")
display(
    bm25_right_dense_wrong[
        ["qid", "question", "relevant_cids", "bm25_pred_parent_ids", "dense_pred_parent_ids"]
    ].head(5)
)

BM25 sai nhưng Dense đúng: 1256
BM25 đúng nhưng Dense sai: 428
Cả BM25 và Dense cùng đúng: 5487

Ví dụ BM25 sai nhưng Dense đúng:


,qid,question,relevant_cids,bm25_pred_parent_ids,dense_pred_parent_ids
10,76459,Khi sát hạch thực hành lái tàu trên đường sắt ...,"[""38367"", ""38367""]","[""38333"", ""123391"", ""38361"", ""82168"", ""38331"",...","[""38361"", ""38355"", ""119588"", ""74527"", ""38360"",..."
15,37695,"Quyền của cơ quan, tổ chức, cá nhân chịu sự gi...","[""57027""]","[""108357"", ""238843"", ""59934"", ""197701"", ""10195...","[""57027"", ""101073"", ""57026"", ""261925"", ""5123"",..."
19,44658,Quy định về mẫu dấu hợp quy đối với phương tiệ...,"[""83496""]","[""232999"", ""33062"", ""197701"", ""78903"", ""41554""...","[""233000"", ""37477"", ""83496"", ""78547"", ""84071"",..."
20,136635,Ngân hàng thương mại kinh doanh bất động sản t...,"[""57545"", ""68706""]","[""60878"", ""235714"", ""156381"", ""201236"", ""79902...","[""57545"", ""112906"", ""91065"", ""157196"", ""71431""..."
30,61471,Số biên chế của Trường Hải quan Việt Nam do ai...,"[""84423""]","[""220358"", ""73739"", ""54779"", ""143960"", ""130040...","[""126383"", ""199383"", ""130336"", ""106423"", ""2600..."



Ví dụ BM25 đúng nhưng Dense sai:


,qid,question,relevant_cids,bm25_pred_parent_ids,dense_pred_parent_ids
7,3429,Thủ tục đăng ký kết hôn gồm những bước nào? Ai...,"[""61763""]","[""78833"", ""78326"", ""183733"", ""47574"", ""4565"", ...","[""61762"", ""183733"", ""65785"", ""72531"", ""61761"",..."
8,102355,"Vụ Giáo dục Chính trị và Công tác học sinh, si...","[""176126""]","[""14385"", ""49790"", ""176126"", ""189311"", ""92472""...","[""235198"", ""53015"", ""72687"", ""173882"", ""92177""..."
23,5601,Khi thi thăng hạng chức danh nghề nghiệp từ Lư...,"[""61923"", ""61923""]","[""61924"", ""148058"", ""77610"", ""36432"", ""14239"",...","[""77610"", ""98844"", ""241769"", ""139208"", ""37057""..."
37,13189,Cán bộ Viện Nghiên cứu Da và Giầy có được đề x...,"[""76185""]","[""245390"", ""76183"", ""245388"", ""86352"", ""76185""...","[""245390"", ""245388"", ""170708"", ""76183"", ""11803..."
53,60784,Định hướng phát triển không gian trong nội dun...,"[""44490""]","[""72460"", ""44496"", ""210107"", ""44490"", ""95468"",...","[""69681"", ""137671"", ""241900"", ""200875"", ""24101..."


# Cell 14 - Lưu phân tích so sánh BM25 và Dense Retrieval

## Mục tiêu cell này
Lưu lại các trường hợp BM25 và Dense Retrieval đúng/sai khác nhau trên validation set.

## Vì sao cần làm bước này?
Kết quả ở Cell 13 cho thấy:
- Dense tìm đúng nhiều câu mà BM25 bỏ sót
- BM25 vẫn tìm đúng một số câu mà Dense bỏ sót
- Hai phương pháp có thế mạnh khác nhau

Đây là lý do quan trọng để xây dựng Hybrid Retrieval ở notebook sau.

## Giải thích code
Code sẽ:
1. Tạo bảng tổng hợp số lượng các nhóm:
   - BM25 sai nhưng Dense đúng
   - BM25 đúng nhưng Dense sai
   - cả hai cùng đúng
   - cả hai cùng sai
2. Lưu bảng tổng hợp vào `dense_vs_bm25_case_summary.csv`
3. Lưu các ví dụ BM25 sai nhưng Dense đúng
4. Lưu các ví dụ BM25 đúng nhưng Dense sai

Các file này sẽ dùng để viết phần phân tích thực nghiệm trong báo cáo.

## Output mong đợi
Bạn cần thấy các file được lưu thành công và bảng tổng hợp 4 nhóm kết quả.

In [14]:
both_wrong = compare_pred_df[
    (compare_pred_df["bm25_hit@10"] == 0) & (compare_pred_df["dense_hit@10"] == 0)
].copy()

case_summary_df = pd.DataFrame([
    {
        "case": "BM25_wrong_Dense_right",
        "count": len(bm25_wrong_dense_right),
        "meaning": "Dense tìm được tài liệu đúng nhưng BM25 không tìm được trong top 10"
    },
    {
        "case": "BM25_right_Dense_wrong",
        "count": len(bm25_right_dense_wrong),
        "meaning": "BM25 tìm được tài liệu đúng nhưng Dense không tìm được trong top 10"
    },
    {
        "case": "Both_right",
        "count": len(both_right),
        "meaning": "Cả BM25 và Dense đều tìm được tài liệu đúng trong top 10"
    },
    {
        "case": "Both_wrong",
        "count": len(both_wrong),
        "meaning": "Cả BM25 và Dense đều không tìm được tài liệu đúng trong top 10"
    }
])

case_summary_path = METRIC_DIR / "dense_vs_bm25_case_summary.csv"
bm25_wrong_dense_right_path = PRED_DIR / "bm25_wrong_dense_right_examples.csv"
bm25_right_dense_wrong_path = PRED_DIR / "bm25_right_dense_wrong_examples.csv"

case_summary_df.to_csv(case_summary_path, index=False, encoding="utf-8-sig")
bm25_wrong_dense_right.to_csv(bm25_wrong_dense_right_path, index=False, encoding="utf-8-sig")
bm25_right_dense_wrong.to_csv(bm25_right_dense_wrong_path, index=False, encoding="utf-8-sig")

print("Đã lưu case summary:", case_summary_path)
print("Đã lưu BM25 sai Dense đúng:", bm25_wrong_dense_right_path)
print("Đã lưu BM25 đúng Dense sai:", bm25_right_dense_wrong_path)

display(case_summary_df)

Đã lưu case summary: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\dense_vs_bm25_case_summary.csv
Đã lưu BM25 sai Dense đúng: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\bm25_wrong_dense_right_examples.csv
Đã lưu BM25 đúng Dense sai: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\bm25_right_dense_wrong_examples.csv


,case,count,meaning
0,BM25_wrong_Dense_right,1256,Dense tìm được tài liệu đúng nhưng BM25 không ...
1,BM25_right_Dense_wrong,428,BM25 tìm được tài liệu đúng nhưng Dense không ...
2,Both_right,5487,Cả BM25 và Dense đều tìm được tài liệu đúng tr...
3,Both_wrong,1434,Cả BM25 và Dense đều không tìm được tài liệu đ...


# Cell 15 - Kiểm tra cuối các file Dense Retrieval đã tạo

## Mục tiêu cell này
Kiểm tra toàn bộ file đầu ra quan trọng của notebook `05_build_dense_index.ipynb`.

## Vì sao cần làm bước này?
Notebook 05 đã tạo dense embeddings, FAISS index và đánh giá Dense Retrieval trên validation set.  
Các notebook sau sẽ cần dùng lại các file này để:
- chạy Dense Retrieval
- xây Hybrid Retrieval bằng cách kết hợp BM25 và Dense
- so sánh hiệu quả giữa BM25, Dense và Hybrid
- dùng dense predictions làm baseline thực nghiệm

Nếu thiếu file ở bước này, notebook sau có thể lỗi hoặc không thể so sánh kết quả.

## Giải thích code
Code sẽ kiểm tra các file:
- `corpus_embeddings.npy`: embedding của toàn bộ chunks
- `faiss_index.index`: FAISS dense vector index
- `dense_metadata.csv`: metadata tương ứng với embeddings
- `dense_val_predictions.csv`: kết quả Dense Retrieval trên validation
- `dense_val_metrics.csv`: metric Dense Retrieval
- `bm25_vs_dense_val_metrics.csv`: bảng so sánh BM25 và Dense
- `dense_vs_bm25_case_summary.csv`: phân tích trường hợp đúng/sai giữa BM25 và Dense

## Output mong đợi
Tất cả file cần có trạng thái `OK`.  
Bảng metric Dense Retrieval và bảng so sánh BM25 vs Dense được hiển thị lại để chốt kết quả.

In [15]:
required_dense_files = [
    FAISS_DIR / "corpus_embeddings.npy",
    FAISS_DIR / "faiss_index.index",
    FAISS_DIR / "dense_metadata.csv",
    PRED_DIR / "dense_val_predictions.csv",
    METRIC_DIR / "dense_val_metrics.csv",
    METRIC_DIR / "bm25_vs_dense_val_metrics.csv",
    METRIC_DIR / "dense_vs_bm25_case_summary.csv"
]

dense_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    }
    for path in required_dense_files
])

dense_val_metrics_df = pd.read_csv(METRIC_DIR / "dense_val_metrics.csv")
bm25_vs_dense_df = pd.read_csv(METRIC_DIR / "bm25_vs_dense_val_metrics.csv")
case_summary_df = pd.read_csv(METRIC_DIR / "dense_vs_bm25_case_summary.csv")

display(dense_check_df)

print("Dense Retrieval validation metrics:")
display(dense_val_metrics_df)

print("BM25 vs Dense comparison:")
display(bm25_vs_dense_df)

print("Case summary:")
display(case_summary_df)

print("Tổng file OK:", (dense_check_df["status"] == "OK").sum(), "/", len(dense_check_df))

,file,status,size_mb
0,indexes\faiss\corpus_embeddings.npy,OK,267.91
1,indexes\faiss\faiss_index.index,OK,267.91
2,indexes\faiss\dense_metadata.csv,OK,182.62
3,artifacts\predictions\dense_val_predictions.csv,OK,2.42
4,artifacts\metrics\dense_val_metrics.csv,OK,0.00
5,artifacts\metrics\bm25_vs_dense_val_metrics.csv,OK,0.00
6,artifacts\metrics\dense_vs_bm25_case_summary.csv,OK,0.00


Dense Retrieval validation metrics:


,metric,value,method
0,recall@1,0.421615,Dense_E5
1,recall@3,0.624869,Dense_E5
2,recall@5,0.700407,Dense_E5
3,recall@10,0.783614,Dense_E5
4,mrr,0.540115,Dense_E5
5,ndcg@10,0.585338,Dense_E5


BM25 vs Dense comparison:


,metric,BM25,Dense_E5,improvement_dense_minus_bm25,relative_improvement_percent
0,mrr,0.453844,0.540115,0.086271,19.01
1,ndcg@10,0.497519,0.585338,0.087819,17.65
2,recall@1,0.343754,0.421615,0.077862,22.65
3,recall@10,0.687391,0.783614,0.096223,14.00
4,recall@3,0.528995,0.624869,0.095874,18.12
5,recall@5,0.605113,0.700407,0.095293,15.75


Case summary:


,case,count,meaning
0,BM25_wrong_Dense_right,1256,Dense tìm được tài liệu đúng nhưng BM25 không ...
1,BM25_right_Dense_wrong,428,BM25 tìm được tài liệu đúng nhưng Dense không ...
2,Both_right,5487,Cả BM25 và Dense đều tìm được tài liệu đúng tr...
3,Both_wrong,1434,Cả BM25 và Dense đều không tìm được tài liệu đ...


Tổng file OK: 7 / 7



## 1. File 05 làm gì?

File 05 tạo một **công cụ tìm kiếm theo ngữ nghĩa**.

File 04 là:

```text
BM25 = tìm theo từ khóa
```

File 05 là:

```text
Dense Retrieval = tìm theo ý nghĩa
```

Ví dụ người dùng hỏi:

```text
Laptop công ty khi nghỉ việc thì xử lý sao?
```

Trong tài liệu handbook có thể không ghi đúng chữ “laptop khi nghỉ việc”, mà ghi kiểu:

```text
Managing work devices
Everyone receives a computer...
```

BM25 có thể yếu nếu không trùng từ khóa.
Dense Retrieval có thể hiểu “laptop công ty” gần nghĩa với “work devices”.

---

## 2. Vì sao cần Dense Retrieval?

Vì người dùng ngoài thực tế thường hỏi **khác lời văn trong tài liệu**.

Ví dụ:

```text
Người dùng hỏi:
Lương thử việc được tính như thế nào?
```

Tài liệu có thể ghi:

```text
Tiền lương trong thời gian thử việc...
```

BM25 tìm theo từ khóa, nếu từ khóa không trùng nhiều thì có thể xếp thấp.

Dense Retrieval biến cả câu hỏi và tài liệu thành vector ngữ nghĩa, nên nó hiểu:

```text
lương thử việc ≈ tiền lương trong thời gian thử việc
```

Đó là lý do file 05 quan trọng.

---

## 3. File 05 đã làm những bước nào?

### Bước 1: Load corpus

Mình load lại:

```text
data/chunks/all_chunks.csv
```

Corpus có:

```text
91,353 legal chunks
95 company handbook chunks
Tổng: 91,448 chunks
```

Đây là kho tài liệu mà Dense Retrieval sẽ tìm kiếm.

---

### Bước 2: Kiểm tra GPU và thư viện

Bạn đã kiểm tra được:

```text
Torch version: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
FAISS version: 1.12.0
```

Nghĩa là máy bạn có GPU và dùng được để tạo embedding nhanh hơn.

---

### Bước 3: Load embedding model

Mình dùng model:

```text
intfloat/multilingual-e5-base
```

Lý do dùng model này:

```text
Nó hỗ trợ nhiều ngôn ngữ
Corpus của mình có tiếng Việt + tiếng Anh
Phù hợp cho retrieval
```

Với E5, mình dùng format:

```text
query: câu hỏi người dùng
passage: nội dung tài liệu
```

Ví dụ:

```text
query: Lương thử việc được quy định như thế nào?
```

và:

```text
passage: Điều 26. Tiền lương trong thời gian thử việc...
```

---

## 4. Embedding là gì?

Embedding là biến văn bản thành vector số.

Ví dụ:

```text
"Lương thử việc được quy định như thế nào?"
```

được biến thành một vector 768 chiều:

```text
[0.012, -0.034, 0.087, ...]
```

Tài liệu cũng được biến thành vector.

Sau đó hệ thống so sánh:

```text
vector câu hỏi gần vector tài liệu nào nhất?
```

Tài liệu càng gần về nghĩa thì điểm similarity càng cao.

---

## 5. File 05 đã tạo embedding cho toàn bộ corpus

Bạn đã encode:

```text
91,448 chunks
```

Kết quả:

```text
Embeddings shape: (91448, 768)
Embeddings size: 267.91 MB
```

Nghĩa là:

```text
Mỗi chunk → 1 vector 768 chiều
91,448 chunks → 91,448 vectors
```

File đã lưu:

```text
indexes/faiss/corpus_embeddings.npy
```

---

## 6. File 05 đã build FAISS index

Sau khi có vector, mình tạo FAISS index:

```text
indexes/faiss/faiss_index.index
```

FAISS giúp tìm vector gần nhất nhanh hơn.

Nếu không có FAISS, mỗi lần hỏi hệ thống phải so sánh câu hỏi với 91,448 vector thủ công, sẽ chậm.

FAISS giống như:

```text
Cơ sở dữ liệu vector để tìm tài liệu gần nghĩa nhanh
```

---

## 7. Dense Search hoạt động thế nào?

Ví dụ câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Dense Retrieval trả về:

```text
Rank 1: Vi phạm quy định về thử việc
Rank 2: Điều 26. Tiền lương trong thời gian thử việc
Rank 3: Bảo hiểm đối với người thử việc
Rank 4: Thử việc và thời gian thử việc
```

Điều này rất hợp lý vì các đoạn này đều liên quan đến “lương thử việc”.

Ví dụ câu hỏi handbook:

```text
What is the company policy for managing work devices?
```

Dense trả về đúng nhóm:

```text
Managing Work Devices
```

---

## 8. Kết quả Dense Retrieval so với BM25

BM25 trước đó:

```text
Recall@10 = 0.6874
MRR = 0.4538
```

Dense Retrieval:

```text
Recall@10 = 0.7836
MRR = 0.5401
```

Nói dễ hiểu:

```text
BM25 tìm đúng tài liệu trong top 10 khoảng 68.7% câu hỏi.
Dense tìm đúng tài liệu trong top 10 khoảng 78.4% câu hỏi.
```

Dense tốt hơn BM25 rõ ràng.

---

## 9. Vì sao Dense tốt hơn?

Vì Dense hiểu nghĩa tốt hơn từ khóa.

Ví dụ trong validation:

```text
BM25 sai nhưng Dense đúng: 1256 câu
BM25 đúng nhưng Dense sai: 428 câu
Cả hai cùng đúng: 5487 câu
Cả hai cùng sai: 1434 câu
```

Điều này nói lên:

```text
Dense mạnh hơn tổng thể.
Nhưng BM25 vẫn có vài câu mà Dense bỏ sót.
```

Vì vậy mình không bỏ BM25, mà bước sau sẽ làm:

```text
Hybrid Retrieval = BM25 + Dense
```

---

## 10. Ví dụ thực tế trong app

Người dùng hỏi:

```text
Công ty nước ngoài có policy về thiết bị làm việc, nếu áp dụng ở Việt Nam thì cần lưu ý gì?
```

Dense Retrieval có thể tìm:

```text
Managing Work Devices trong handbook
```

và tìm thêm các đoạn pháp luật Việt Nam liên quan nếu câu hỏi có yếu tố pháp lý.

Hoặc người dùng hỏi:

```text
Chính sách thử việc của công ty có cần đối chiếu quy định Việt Nam không?
```

Dense Retrieval tìm các đoạn legal liên quan đến:

```text
tiền lương thử việc
thời gian thử việc
vi phạm quy định về thử việc
```

Sau đó RAG mới đưa các context này cho LLM trả lời có nguồn.

---

## 11. File 05 đã tạo những file nào?

Các file quan trọng:

```text
indexes/faiss/corpus_embeddings.npy
indexes/faiss/faiss_index.index
indexes/faiss/dense_metadata.csv
artifacts/predictions/dense_val_predictions.csv
artifacts/metrics/dense_val_metrics.csv
artifacts/metrics/bm25_vs_dense_val_metrics.csv
artifacts/metrics/dense_vs_bm25_case_summary.csv
```

Tất cả đã OK.

---

## 12. Chốt lại file 05

File 05 làm nhiệm vụ:

```text
Tạo công cụ tìm kiếm theo ngữ nghĩa cho RAG.
```

Nếu file 04 là:

```text
Tìm bằng từ khóa
```

thì file 05 là:

```text
Tìm bằng ý nghĩa
```

Kết quả cho thấy Dense Retrieval tốt hơn BM25, nhưng BM25 vẫn có ích. Vì vậy hướng tiếp theo rất hợp lý:

```text
File 06: Naive RAG dùng Dense Retrieval để tạo bản RAG đầu tiên
File 07: Hybrid RAG kết hợp BM25 + Dense
```
